# 08 — Assets on macro factors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamadouyamar/CVineMarketGen/blob/main/examples/08_assets_on_macro_factors.ipynb)

The seven macro factors of notebook 06 are simulated by the C-vine. An asset is
then `alpha + beta' f + eps`: its betas on the factors, estimated on its history,
map every factor scenario to an asset return. A view on the factors becomes a
distribution for every asset through its betas.

Runtime: about 5 minutes.

## Import

In [1]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "examples" else os.getcwd()
sys.path.insert(0, ROOT)                 # run from a local clone, from examples/ or the root
try:
    import cvinemarketgen                # local clone or already installed
except ImportError:                      # e.g. on Google Colab
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/mamadouyamar/CVineMarketGen.git"], check=True)
    import cvinemarketgen
print("cvinemarketgen", cvinemarketgen.__version__)

from cvinemarketgen import load_factor_data, load_etf_monthly, Targets, CVineMarket, FactorModel

cvinemarketgen 0.2.0


## Load the seven factors and ten ETFs

Monthly, from the caches of notebook 06 (factors) and Yahoo Finance (ETF
adjusted closes). The ETFs are chosen so that each has an obvious loading:
IWM on the small-cap premium, TIP and TLT on the real premia, HYG and LQD on credit.

The ETF returns are monthly simple returns of the adjusted closes, $r_{j,t} = P_{j,t} / P_{j,t-1} - 1$, and the factors $\mathbf f_t = (f_{1,t}, \dots, f_{K,t})'$, $K = 7$, are the monthly excess returns of notebook 06. The regression below uses the $n$ months common to both tables.

In [2]:
factors = load_factor_data(cache=os.path.join(ROOT, "data", "factors_cache.csv"))
etfs = load_etf_monthly(["SPY", "IWM", "EFA", "EEM", "TLT", "TIP", "LQD", "HYG", "GLD", "VNQ"],
                        cache=os.path.join(ROOT, "data", "etf_cache.csv"))
common = factors.index.intersection(etfs.index)
print(f"factors {factors.shape}, ETFs {etfs.shape}, common window {common.min()} to {common.max()} ({len(common)} months)")

factor data read from cache /Users/mamadouthioub/Desktop/CopulaGenerator/CVineMarketGen/data/factors_cache.csv: 244 months, 2006-04 to 2026-07
ETF returns read from cache /Users/mamadouthioub/Desktop/CopulaGenerator/CVineMarketGen/data/etf_cache.csv: 232 months, 2007-06 to 2026-09
factors (244, 7), ETFs (232, 10), common window 2007-06 to 2026-07 (230 months)


## Fit the factor model

One least-squares regression per ETF on a constant and the seven factors.
`report` holds alpha, the betas, their Newey-West t-statistics, the R² and the
residual volatility, all in monthly units.

For asset $j$ the model is $r_{j,t} = \alpha_j + \boldsymbol\beta_j' \mathbf f_t + \varepsilon_{j,t}$. In matrix form, with $\mathbf X = [\mathbf 1, \mathbf F]$ the $n \times (K+1)$ design and $\mathbf r_j$ the $n$-vector of the asset's returns,

$$
(\hat\alpha_j, \hat{\boldsymbol\beta}_j')' = (\mathbf X'\mathbf X)^{-1} \mathbf X' \mathbf r_j, \qquad \mathbf e_j = \mathbf r_j - \mathbf X (\hat\alpha_j, \hat{\boldsymbol\beta}_j')', \qquad R_j^2 = 1 - \frac{\mathbf e_j'\mathbf e_j}{\sum_t (r_{j,t} - \bar r_j)^2}.
$$

The standard errors are Newey-West: with $\mathbf g_t = \mathbf x_t\, e_{j,t}$ and Bartlett weights $w_\ell = 1 - \ell/(L+1)$,

$$
\hat{\mathbf V} = \frac{1}{n}\, \hat{\mathbf A}^{-1} \hat{\mathbf S}\, \hat{\mathbf A}^{-1}, \qquad \hat{\mathbf A} = \frac{1}{n}\mathbf X'\mathbf X, \qquad \hat{\mathbf S} = \frac{1}{n}\sum_t \mathbf g_t \mathbf g_t' + \frac{1}{n}\sum_{\ell=1}^{L} w_\ell \sum_t \big(\mathbf g_t \mathbf g_{t-\ell}' + \mathbf g_{t-\ell} \mathbf g_t'\big),
$$

with $L = \lfloor 4\,(n/100)^{2/9} \rfloor$ lags. The residual volatility is $\hat\sigma_{e_j} = \big(\frac{1}{n - K - 1}\, \mathbf e_j'\mathbf e_j\big)^{1/2}$, and a Johnson SU with mean $0$, volatility $\hat\sigma_{e_j}$ and the residual's sample skewness and kurtosis is fitted per asset for the simulation step. `report` shows $\hat\alpha_j$, $\hat{\boldsymbol\beta}_j$, $R_j^2$ and $\hat\sigma_{e_j}$, all monthly.

In [3]:
fm = FactorModel(etfs, factors).fit()
fm.report[["alpha"] + fm.factors + ["R2", "resid vol"]].round(3)

,alpha,Equity DM,Equity EM,Real premia,Inflation,Credit,Commodity,Small cap,R2,resid vol
SPY,0.003,0.973,-0.161,-0.041,0.084,-0.036,-0.035,-0.056,0.950,0.010
IWM,0.002,1.005,-0.161,-0.109,0.003,-0.007,0.001,0.969,0.951,0.013
EFA,-0.001,1.085,0.140,0.000,-0.385,-0.075,0.052,-0.116,0.922,0.014
EEM,0.000,1.091,0.988,-0.018,-0.199,-0.178,0.040,0.060,0.946,0.014
TLT,0.005,-0.042,-0.022,1.860,-2.353,0.365,-0.022,0.002,0.910,0.012
TIP,0.002,0.014,0.010,0.856,-0.102,0.049,0.031,-0.007,0.903,0.005
LQD,0.003,0.154,0.001,0.968,-0.949,0.430,-0.017,0.012,0.798,0.011
HYG,0.002,0.345,-0.003,0.391,-0.242,0.353,0.012,0.104,0.674,0.017
GLD,0.009,-0.088,0.383,1.308,-0.453,-0.021,0.278,-0.182,0.380,0.040
VNQ,0.001,0.921,-0.125,0.657,-1.099,0.474,0.001,0.320,0.616,0.040


## Read the betas

t-statistics above 2 in absolute value are the loadings that matter. Check the
expected ones: IWM on Small cap, TIP on Real premia, HYG and LQD on Credit, EEM on
Equity EM. TLT loads on Real premia and negatively on Inflation: the Inflation
factor is long breakevens, which hurts nominal bonds; TIP, whose price follows
the real yield, has no significant Inflation loading once Real premia is in.

The $t$-statistic of each coefficient is the estimate over its Newey-West standard error, $t_{jk} = \hat\beta_{jk} / \sqrt{\hat V_{kk}}$; with $n$ around 200 months, $|t| > 2$ is the usual 5 percent threshold.

In [4]:
fm.tstat.round(1)

,alpha,Equity DM,Equity EM,Real premia,Inflation,Credit,Commodity,Small cap
SPY,4.6,44.5,-7.4,-1.0,1.2,-0.6,-2.4,-2.0
IWM,2.2,30.8,-4.8,-1.9,0.0,-0.1,0.0,30.0
EFA,-1.2,35.4,4.0,0.0,-3.7,-1.5,2.5,-2.6
EEM,0.4,32.0,24.2,-0.3,-2.2,-3.0,1.6,1.7
TLT,6.9,-1.0,-0.7,24.8,-24.6,4.0,-1.0,0.1
TIP,8.1,0.7,0.9,37.2,-1.2,1.7,2.7,-0.5
LQD,5.5,5.3,0.1,13.9,-11.5,5.2,-0.7,0.5
HYG,2.3,7.5,-0.1,5.5,-2.3,5.2,0.4,1.9
GLD,3.9,-0.8,4.0,7.6,-1.9,-0.1,3.8,-1.9
VNQ,0.5,6.7,-1.4,3.0,-5.1,3.4,0.0,2.9


$R_j^2$ is the share of the asset's monthly variance explained by the factors, $R_j^2 = \mathrm{Var}(\hat{\boldsymbol\beta}_j'\mathbf f_t) / \mathrm{Var}(r_{j,t})$ in sample; $1 - R_j^2$ is the share carried by the residual $\varepsilon_{j,t}$.

In [5]:
fm.r2.round(2).to_frame("R2").T

,SPY,IWM,EFA,EEM,TLT,TIP,LQD,HYG,GLD,VNQ
R2,0.95,0.95,0.92,0.95,0.91,0.9,0.8,0.67,0.38,0.62


## Targets for the factors: the sample

No view yet; the factor history is the target, as in notebook 06.

The factor targets are the sample moments of the factor history: $\mu_k = \bar f_k$, $\sigma_k = s_k$, $\gamma_{1,k}, \gamma_{2,k}$ the sample skewness and kurtosis, $\Sigma^*$ the sample correlation, monthly.

In [6]:
t = Targets.from_history(factors)
t.summary();

Targets for 7 assets (returns layer, history frequency M)
               mean     vol    skew     kurt
Equity DM    0.0066  0.0455 -0.6413   4.7346
Equity EM   -0.0006  0.0311 -0.0237   2.9870
Real premia  0.0007  0.0179 -0.6329   5.4048
Inflation    0.0017  0.0135  0.1327   9.5613
Credit       0.0021  0.0237 -1.9096  17.5847
Commodity    0.0020  0.0543 -0.5306   4.9626
Small cap   -0.0007  0.0248  0.3184   3.0306

correlation matrix:
             Equity DM  Equity EM  Real premia  Inflation  Credit  Commodity  Small cap
Equity DM         1.00       0.10         0.30       0.51    0.61       0.51       0.28
Equity EM         0.10       1.00         0.13       0.18    0.21       0.25      -0.02
Real premia       0.30       0.13         1.00       0.21   -0.02       0.11       0.01
Inflation         0.51       0.18         0.21       1.00    0.56       0.58       0.20
Credit            0.61       0.21        -0.02       0.56    1.00       0.44       0.24
Commodity         0.51       0.25

## Simulate the factors with the C-vine, families from the history

The C-vine market of notebook 06: Johnson SU marginals $q_k(z) = \xi_k + \lambda_k \sinh((z - \gamma_k)/\delta_k)$ on the four moments, families selected per edge from the exceedance-correlation signature on the factor history, and parameters calibrated so that the simulated correlations match $\Sigma^*$, $\hat{\boldsymbol\theta}_k = \arg\min \sum_{i<k} (\mathrm{Corr}(F_k^{-1}(u_k), F_i^{-1}(u_i)) - \Sigma^*_{ki})^2$.

In [7]:
t0 = time.time()
cv = CVineMarket(t, central="Equity DM", families="auto").fit()
print(f"fit: {time.time() - t0:.0f} s")
cv.edges

fit: 173 s


,tree,edge,selected family,parameters
0,1,"Equity EM , Equity DM",gumbel 180°,theta=1.057
1,1,"Real premia , Equity DM",gumbel 180°,theta=1.218
2,1,"Inflation , Equity DM",gumbel 180°,theta=1.503
3,1,"Credit , Equity DM",gumbel 180°,theta=1.639
4,1,"Commodity , Equity DM",gumbel 180°,theta=1.471
5,1,"Small cap , Equity DM",gumbel 0°,theta=1.226
6,2,"Real premia , Equity EM | Equity DM",clayton 180°,theta=0.140
7,2,"Inflation , Equity EM | Equity DM",clayton 180°,theta=0.199
8,2,"Credit , Equity EM | Equity DM",gumbel 0°,theta=1.158
9,2,"Commodity , Equity EM | Equity DM",gumbel 0°,theta=1.169


$25{,}000$ factor scenarios $\mathbf F$ from Algorithm 5 (marginals re-fitted on the draw, acceptance within $2 \times 10^{-4}$ on means and volatilities, $5 \times 10^{-2}$ on skewness and kurtosis, $\tau$ on correlations); the diagnostics compare the draw's moments and correlations with the targets.

In [8]:
Fsim = cv.simulate(25000, seed=1)
cv.diagnostics(Fsim).summary();

25000 simulated observations (returns layer)
             mean target  mean simulated  mean diff  vol target  vol simulated  vol diff  skew target  skew simulated  skew diff  kurt target  kurt simulated  kurt diff
Equity DM         0.0066          0.0066       -0.0      0.0455         0.0455       0.0      -0.6413         -0.6413    -0.0000       4.7346          4.7352     0.0006
Equity EM        -0.0006         -0.0006       -0.0      0.0311         0.0311      -0.0      -0.0237          0.0209     0.0446       2.9870          3.0090     0.0220
Real premia       0.0007          0.0007       -0.0      0.0179         0.0179       0.0      -0.6329         -0.6329    -0.0000       5.4048          5.4055     0.0007
Inflation         0.0017          0.0017       -0.0      0.0135         0.0135       0.0       0.1327          0.1327     0.0000       9.5613          9.5629     0.0016
Credit            0.0021          0.0021       -0.0      0.0237         0.0237       0.0      -1.9096         

## Map the factor scenarios to the assets

`fm.simulate` applies alpha and the betas to every row of factor scenarios and
adds, per asset, an independent Johnson SU residual fitted on the regression
residuals. The simulated assets should have the history's mean and volatility.

The simulation map applies the estimated equation row by row,

$$
X_j = \hat\alpha_j + \hat{\boldsymbol\beta}_j' \mathbf F + e_j, \qquad e_j \sim \text{Johnson SU}\big(0,\ \hat\sigma_{e_j},\ \hat\gamma_{1}(e_j),\ \hat\gamma_{2}(e_j)\big),
$$

with $e_j$ drawn independently across assets and independently of $\mathbf F$. Its first two moments follow from those of the factors: $E[X_j] = \hat\alpha_j + \hat{\boldsymbol\beta}_j' \boldsymbol\mu_f$ and $\mathrm{Var}(X_j) = \hat{\boldsymbol\beta}_j' \boldsymbol\Sigma_f \hat{\boldsymbol\beta}_j + \hat\sigma_{e_j}^2$, which equal the asset's sample mean and variance when $\boldsymbol\mu_f$ and $\boldsymbol\Sigma_f$ are the sample's, as here. What the map drops is the correlation between the residuals of different assets (0.68 between SPY and IWM, for instance), which the sample has and the independence of the $e_j$ does not.

In [9]:
X = fm.simulate(Fsim, seed=1)
pd.DataFrame({"history mean": etfs.loc[common].mean(), "simulated mean": X.mean(),
              "history vol": etfs.loc[common].std(), "simulated vol": X.std()}).round(4)

,history mean,simulated mean,history vol,simulated vol
SPY,0.0095,0.0098,0.0449,0.0437
IWM,0.0083,0.0081,0.0594,0.0584
EFA,0.0050,0.0054,0.0503,0.0493
EEM,0.0054,0.0062,0.0610,0.0601
TLT,0.0032,0.0031,0.0408,0.0401
TIP,0.0030,0.0030,0.0167,0.0165
LQD,0.0036,0.0038,0.0238,0.0236
HYG,0.0044,0.0043,0.0295,0.0294
GLD,0.0088,0.0092,0.0505,0.0510
VNQ,0.0066,0.0068,0.0643,0.0641


## With and without the residual

`residuals=False` keeps the factor exposure only. The volatility then drops to
`sqrt(R²)` of the asset's volatility: the gap is the part of the asset the
factors do not explain.

Without the residual, $X_j^0 = \hat\alpha_j + \hat{\boldsymbol\beta}_j' \mathbf F$ and $\mathrm{Var}(X_j^0) = \hat{\boldsymbol\beta}_j' \boldsymbol\Sigma_f \hat{\boldsymbol\beta}_j = R_j^2\, \mathrm{Var}(r_j)$ in sample, so the volatility of the exposure alone is $\sqrt{R_j^2}$ times the asset's; the table checks the identity on the draw.

In [10]:
X0 = fm.simulate(Fsim, residuals=False)
pd.DataFrame({"history": etfs.loc[common].std(), "with residual": X.std(), "exposure only": X0.std(),
              "sqrt(R2) x history": np.sqrt(fm.r2) * etfs.loc[common].std()}).round(4)

,history,with residual,exposure only,sqrt(R2) x history
SPY,0.0449,0.0437,0.0427,0.0437
IWM,0.0594,0.0584,0.0569,0.0580
EFA,0.0503,0.0493,0.0472,0.0483
EEM,0.0610,0.0601,0.0584,0.0593
TLT,0.0408,0.0401,0.0382,0.0389
TIP,0.0167,0.0165,0.0157,0.0159
LQD,0.0238,0.0236,0.0210,0.0213
HYG,0.0295,0.0294,0.0238,0.0242
GLD,0.0505,0.0510,0.0308,0.0311
VNQ,0.0643,0.0641,0.0496,0.0505


## A projection on the factors

A view is a change of the factor means. Here developed equity is set to 0.3 %
per month and credit to −0.2 %; volatilities, correlations and the history
(for the higher moments and the copula families) are unchanged. The implied
expected return of every ETF follows from its betas.

A view is a new factor mean vector $\boldsymbol\mu_f'$ with everything else unchanged. Because the map is affine in $\mathbf F$, the implied asset mean shifts exactly by $\Delta\mu_j = \hat{\boldsymbol\beta}_j' (\boldsymbol\mu_f' - \boldsymbol\mu_f)$, and `implied_mean` returns $\hat\alpha_j + \hat{\boldsymbol\beta}_j' \boldsymbol\mu_f'$. Here Equity DM is set to $0.3$ percent a month and Credit to $-0.2$; the volatilities, correlations and higher moments stay at the sample's.

In [11]:
proj = t.mean.copy()
proj["Equity DM"] = 0.003
proj["Credit"] = -0.002
t2 = Targets(mean=proj, vol=t.vol, corr=t.corr, history=factors)
pd.DataFrame({"sample view": fm.implied_mean(t.mean), "projection": fm.implied_mean(t2.mean)}).round(4)

,sample view,projection
SPY,0.0097,0.0063
IWM,0.0082,0.0046
EFA,0.0054,0.0018
EEM,0.0063,0.0030
TLT,0.0033,0.0019
TIP,0.0030,0.0027
LQD,0.0037,0.0013
HYG,0.0045,0.0018
GLD,0.0091,0.0095
VNQ,0.0068,0.0015


The factors are simulated again under the view. The Johnson SU marginal enters the mean as a pure location term, $X_k = \mu_k + \sigma_k q_k(Z)$, so the only change in the draw is the shift of the two means; the copula, calibrated to the same $\Sigma^*$ on the same history, is the same. The asset scenarios follow through the map, and their quantiles show the shift and the dispersion it leaves untouched.

In [12]:
cv2 = CVineMarket(t2, central="Equity DM", families="auto").fit()
X2 = fm.simulate(cv2.simulate(25000, seed=2), seed=2)
X2.quantile([0.05, 0.5, 0.95]).round(3)

,SPY,IWM,EFA,EEM,TLT,TIP,LQD,HYG,GLD,VNQ
0.05,-0.070,-0.092,-0.083,-0.099,-0.066,-0.025,-0.037,-0.046,-0.076,-0.105
0.50,0.009,0.006,0.005,0.006,0.003,0.004,0.002,0.003,0.010,0.004
0.95,0.072,0.098,0.076,0.096,0.065,0.027,0.037,0.046,0.091,0.099


## Asset paths from factor paths

Factor paths (independent months here) give asset paths through the same map;
`terminal()` is the cumulative return at the horizon.

Factor paths of 12 independent months (no dynamics here, so each month is an i.i.d. draw from the vine) are mapped month by month, $X^{(s)}_{j,t} = \hat\alpha_j + \hat{\boldsymbol\beta}_j' \mathbf F^{(s)}_t + e^{(s)}_{j,t}$, and the terminal return of a path is $\prod_{t=1}^{12} (1 + X^{(s)}_{j,t}) - 1$; its quantiles across the $2{,}000$ paths are the one-year distribution of each asset under the sample view.

In [13]:
P = cv.simulate_paths(2000, 12, seed=3)
PX = fm.simulate(P, seed=3)
print(PX.array.shape)
PX.terminal().quantile([0.05, 0.5, 0.95]).round(3)

(2000, 12, 10)


,SPY,IWM,EFA,EEM,TLT,TIP,LQD,HYG,GLD,VNQ
0.05,-0.144,-0.232,-0.216,-0.267,-0.196,-0.060,-0.092,-0.117,-0.182,-0.277
0.50,0.115,0.085,0.063,0.064,0.038,0.037,0.041,0.052,0.102,0.067
0.95,0.412,0.491,0.386,0.473,0.289,0.132,0.189,0.233,0.476,0.510


## Save

The saved factor model holds $\hat\alpha_j$, $\hat{\boldsymbol\beta}_j$, the standard errors, $R_j^2$, $\hat\sigma_{e_j}$ and the residual Johnson SU parameters; reloading it reproduces the same map, checked here on the betas.

In [14]:
fm.save("factor_model.json")
cv.save("factor_market.json")
fm2 = FactorModel.load("factor_model.json")
np.allclose(fm2.beta.values, fm.beta.values)

True

## Where to go next

Notebook 06 builds the factors and compares the two generators on them;
notebook 05 targets a published table of assumptions instead of a history.
The same `FactorModel` takes your own asset series in place of the ETFs.